# Moses で統計的機械翻訳システムを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_machine_translation/moses_smt.ipynb)

NLP クックブック 26-08 ／ 機械翻訳

ニューラル以前の主流だったフレーズベース統計的機械翻訳 (SMT) を、対訳コーパスから自分で組み立てたい。単語アライメント、フレーズテーブル、言語モデル、デコードまでを一通り動かす。

In [1]:
# Moses 本体と、単語アライメントの fast_align をビルドする (Colab で 15 分ほどかかる)
!apt-get install -y -q build-essential cmake libboost-all-dev zlib1g-dev libbz2-dev liblzma-dev > /dev/null 2>&1
!git clone -q --depth 1 https://github.com/moses-smt/mosesdecoder
!cd mosesdecoder && ./bjam -j4 -q > ../bjam.log 2>&1; tail -n 1 ../bjam.log
!git clone -q --depth 1 https://github.com/clab/fast_align
!mkdir -p fast_align/build && cd fast_align/build && cmake .. > /dev/null 2>&1 && make -j4 > /dev/null 2>&1
!ls mosesdecoder/bin/moses mosesdecoder/bin/lmplz fast_align/build/fast_align fast_align/build/atools

SUCCESS
fast_align/build/atools      mosesdecoder/bin/lmplz
fast_align/build/fast_align  mosesdecoder/bin/moses


In [2]:
!pip install -q datasets fugashi unidic-lite sacremoses sacrebleu

In [3]:
import os
from datasets import load_dataset
from fugashi import Tagger
from sacremoses import MosesTokenizer

os.makedirs("data", exist_ok=True)
ds = load_dataset("Helsinki-NLP/opus-100", "en-ja")       # 映画・ドラマの字幕が中心の対訳データ
tagger, mt = Tagger("-Owakati"), MosesTokenizer(lang="en")

def tok_ja(s):
    s = s.translate(str.maketrans("|<>[]　", "｜＜＞［］ "))     # Moses は | < > [ ] を特別な文字として扱うので、全角に置き換える
    return tagger.parse(s).strip()

def tok_en(s):
    return mt.tokenize(s.lower(), escape=True, return_str=True)    # 小文字にそろえて単語に分ける

def write(split, n, prefix, max_len=40):
    count = 0
    with open(f"data/{prefix}.ja", "w") as f_ja, open(f"data/{prefix}.en", "w") as f_en, open(f"data/{prefix}.raw.en", "w") as f_raw:
        for ex in ds[split]:
            ja, en = ex["translation"]["ja"].strip(), ex["translation"]["en"].strip()
            if not ja or not en or len(ja) > 60:
                continue
            tj, te = tok_ja(ja), tok_en(en)
            if not (1 <= len(tj.split()) <= max_len and 1 <= len(te.split()) <= max_len):     # 長すぎる文はアライメントの質を下げるので除く
                continue
            f_ja.write(tj + "\n"); f_en.write(te + "\n"); f_raw.write(en.lower() + "\n")
            count += 1
            if count >= n:
                break
    print(prefix, count, "文対")

write("train", 50000, "train")
write("test", 500, "test")
!head -n 3 data/train.ja data/train.en

train 50000 文対
test 500 文対
==> data/train.ja <==
- ラウール - ラウール に ヴィンセント ・ ハンナ だ
いま 地下 に 居 ます 他 の 2 人 と 一緒 に 誘拐 さ れ た ん です ！
動い た よ ！

==> data/train.en <==
yeah , vincent hanna .
i &apos;m being held in a basement. i &apos;ve been abducted with two other girls .
it works !


In [4]:
# 1. 単語アライメント: 日→英と英→日の両方向で求め、grow-diag-final-and で 1 つにまとめる
!paste -d '\t' data/train.ja data/train.en | sed 's/\t/ ||| /' > data/train.ja-en
!fast_align/build/fast_align -i data/train.ja-en -d -o -v > data/fwd.align 2> /dev/null
!fast_align/build/fast_align -i data/train.ja-en -d -o -v -r > data/rev.align 2> /dev/null
!fast_align/build/atools -i data/fwd.align -j data/rev.align -c grow-diag-final-and > data/aligned.grow-diag-final-and
!head -n 3 data/aligned.grow-diag-final-and

# 2. 言語モデル: 訳文側 (英語) の 3-gram を KenLM で学習する
!mosesdecoder/bin/lmplz -o 3 --discount_fallback < data/train.en > data/lm.arpa 2> /dev/null
!mosesdecoder/bin/build_binary data/lm.arpa data/lm.blm > /dev/null 2>&1

0-0 1-2 2-2 3-2 4-1 5-2 6-1 7-3 8-4
0-0 0-1 1-4 2-3 3-4 4-4 5-13 6-5 7-12 8-12 9-11 10-11 11-11 12-10 13-10 14-9 15-8 16-14 17-15 18-15
0-0 1-1 2-2 3-2


In [5]:
# 3. 翻訳モデル: アライメントからフレーズ対を抽出し、翻訳確率と並べ替えモデルを推定する (手順 4〜9)
!mkdir -p train/model && cp data/aligned.grow-diag-final-and train/model/
!mosesdecoder/scripts/training/train-model.perl --root-dir train --corpus data/train --f ja --e en \
    --alignment grow-diag-final-and --reordering msd-bidirectional-fe --lm 0:3:$PWD/data/lm.blm:8 \
    --first-step 4 --last-step 9 --cores 4 > train.log 2>&1
!ls -lh train/model/

# フレーズテーブルの中身: 「原言語のフレーズ ||| 目的言語のフレーズ ||| 4 つの確率 ||| アライメント ||| 頻度」
import gzip
from collections import defaultdict

table = defaultdict(list)
with gzip.open("train/model/phrase-table.gz", "rt") as f:
    for line in f:
        src, tgt, scores = line.split(" ||| ")[:3]
        if src in ("ありがとう", "分から ない", "お 願い"):
            table[src].append((float(scores.split()[2]), tgt))      # 3 番目のスコアが P(英語 | 日本語)
for src, cands in table.items():
    print(src, "→", [(tgt, round(p, 3)) for p, tgt in sorted(cands, reverse=True)[:6]])

total 46M
-rw-r--r-- 1 root root 2.0M Nov 15 07:03 aligned.grow-diag-final-and
-rw-r--r-- 1 root root 6.4M Nov 15 07:04 extract.inv.sorted.gz
-rw-r--r-- 1 root root 5.3M Nov 15 07:04 extract.o.sorted.gz
-rw-r--r-- 1 root root 6.2M Nov 15 07:04 extract.sorted.gz
-rw-r--r-- 1 root root 3.3M Nov 15 07:03 lex.e2f
-rw-r--r-- 1 root root 3.3M Nov 15 07:03 lex.f2e
-rw-r--r-- 1 root root 1.1K Nov 15 07:04 moses.ini
-rw-r--r-- 1 root root  14M Nov 15 07:04 phrase-table.gz
-rw-r--r-- 1 root root 5.6M Nov 15 07:04 reordering-table.wbe-msd-bidirectional-fe.gz
ありがとう → [('thank you', 0.196), ('thanks', 0.174), ('thank', 0.087), ('thanks .', 0.043), ('thanks ,', 0.043), ('.', 0.043)]
お 願い → [('please', 0.268), ('please .', 0.124), ('please ,', 0.052), (', please .', 0.031), ('thank you', 0.021), ('me', 0.021)]
分から ない → [('don &apos;t know', 0.103), ('i don &apos;t know .', 0.077), ('i don &apos;t know', 0.077), ('i don &apos;t understand .', 0.051), ('don &apos;t', 0.051), ('.', 0.051)]


In [6]:
# 4. デコード: テストデータを翻訳して BLEU を測る
!mosesdecoder/bin/moses -f train/model/moses.ini -threads 4 < data/test.ja > data/test.out 2> decode.log

import sacrebleu
from sacremoses import MosesDetokenizer

md = MosesDetokenizer(lang="en")
hyps = [md.detokenize(line.split()) for line in open("data/test.out")]
refs = [line.strip() for line in open("data/test.raw.en")]
print(sacrebleu.corpus_bleu(hyps, [refs]))
for src, hyp, ref in list(zip(open("data/test.ja"), hyps, refs))[:8]:
    print(f"{src.strip()}\n   訳: {hyp}\n   参照: {ref}")

BLEU = 5.50 38.8/9.7/4.4/2.2 (BP = 0.708 ratio = 0.743 hyp_len = 3588 ref_len = 4827)
もし そう なっ たら
   訳: if i'd
   参照: if what i'm saying is true,
お 父 さん か ？ 悪い けど 、 壁 が ある ん だ けど
   訳: your father? i'm sorry, but you have the wall.
   参照: you didn't think you got your fighting skills from your father, did you?
フリスビー 知っ てる ？
   訳: フリスビー?
   参照: you think a dog knows how to work a frisbee?
オフィス へ
   訳: his office.
   参照: so you lied to me about a case?
農夫 は 春 に とうもろこし の 種 を まく 。
   訳: 農夫 is in spring とうもろこし pabulum the まく.
   参照: the farmer plants the corn in the spring.
期待 に 胸 膨らま せ て
   訳: i only 膨らま him.
   参照: to where i am expected, ma'am
彼女 の 成功 で 私 も 同じ 事 を やっ て みよう と いう 気 に なっ た 。
   訳: she works at me the same thing to do you see that.
   参照: her success encouraged me to try the same thing.
彼 ら が デート を し て いる の を 見かけ て 「 近々 結婚 する の 」 って きい た ん だ 。
   訳: they did to have you seen "近々 married the" きい.
   参照: when i saw them on a date, i asked them, "do i hear wedding bells?"


In [7]:
# 翻訳の過程を見る: どのフレーズ対を使って訳したか (-t でフレーズの対応を出力する)
!echo "ありがとう 。 本当 に 助かり まし た 。" | mosesdecoder/bin/moses -f train/model/moses.ini -t 2> /dev/null
!echo "彼 は どこ に 行っ た の か 分から ない 。" | mosesdecoder/bin/moses -f train/model/moses.ini -t 2> /dev/null

thank you |0-0| for |1-3| 助かり |4-4| . |5-7| 
where is he |0-2| went |3-5| you |6-7| don &apos;t know . |8-10| 


- フレーズベース SMT は「翻訳モデル（フレーズ対とその確率）」「言語モデル（訳文の自然さ）」「並べ替えモデル」を別々に学習し、デコーダがそれらのスコアの重み付き和が最大になる訳を探す。ニューラル機械翻訳と違い、どのフレーズ対を使ったかを直接たどれる。
- 5 万文対・チューニングなしで、字幕データでの BLEU は 5 前後（実測は 5.4〜5.5。実行する環境によって単語アライメントの結果がわずかに変わり、値も少し揺れる）。語順が大きく違う日英は、フレーズベースの手法が特に苦手とする言語対。
- 学習データにない単語（未知語）は、訳されずにそのまま出力される。データが少ないと未知語が多くなる。
- ここでは省略したが、本来は開発データで各モデルの重みを調整する（MERT。`scripts/training/mert-moses.pl`）。BLEU が数ポイント変わるので、きちんと比較するなら必須。
- 単語アライメントには GIZA++ が標準的に使われてきたが、fast_align のほうがはるかに速く、質も大きくは変わらない。`train-model.perl` の手順 1〜3（GIZA++ によるアライメント）を飛ばして、手順 4 から実行している。
- ビルドに時間がかかる（4 コアで約 6 分、Colab の 2 コアではその 2〜3 倍）。うまくいかないときは `bjam.log` と `train.log` を確認する。
- 現在の実用システムはほぼニューラルに置き換わったが、フレーズテーブルは対訳辞書の抽出や訳語の分析に、単語アライメントは用語の対応付けや注釈の投影に、今も使われている。